In [2]:
import pandas as pd
df = pd.read_csv("../data/processed/cleaned_data.csv")

Build each heuristic feature


Duplicate / near-duplicate content (spammy accounts often reuse text):


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#dropping rows with missing content_clean values 
df = df.dropna(subset=["content_clean"])
 
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['content_clean'])
 
# Exact duplicate flag (fast)
df['is_exact_dup'] = df.duplicated(subset=['content_clean'], keep=False).astype(int)
 
# Near-duplicate flag via cosine similarity (do this in batches if dataset is large — full pairwise
# similarity is O(n^2) and will blow up memory past ~20k rows; sample or use nearest-neighbors instead)
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=2, metric='cosine').fit(X)
distances, indices = nn.kneighbors(X)
df['near_dup_score'] = 1 - distances[:, 1]   # similarity to nearest OTHER review
df['is_near_dup'] = (df['near_dup_score'] > 0.9).astype(int)


Burstiness (spike in review volume in a short window):


In [5]:
print(df["at"].dtype)

str


In [6]:
df["at"] = pd.to_datetime(df["at"], errors="coerce")
daily_counts = df.groupby(df['at'].dt.date).size()
mean_c, std_c = daily_counts.mean(), daily_counts.std()
burst_days = daily_counts[daily_counts > mean_c + 2*std_c].index  # >2 std above average
df['is_burst_day'] = df['at'].dt.date.isin(burst_days).astype(int)



One-hit-wonder accounts:


In [7]:
user_counts = df['userName'].value_counts()
df['user_review_count'] = df['userName'].map(user_counts)
df['is_one_hit_wonder'] = (df['user_review_count'] == 1).astype(int)


Rating deviation from consensus (per app version, since that's your closest thing to "product"):


In [8]:
version_mean = df.groupby('reviewCreatedVersion')['score'].transform('mean')
df['rating_deviation'] = (df['score'] - version_mean).abs()



Linguistic features

In [9]:
df['exclamation_count'] = df['content'].str.count('!')
df['word_count'] = df['content_clean'].str.split().str.len()
df['avg_word_len'] = df['content_clean'].apply(lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0)
df['superlative_count'] = df['content_clean'].str.count(r'\b(best|worst|amazing|terrible|perfect|awful)\b')


Combine into one pseudo-label


In [10]:
# Normalize each signal to 0-1, then take a weighted sum. Weights are a judgment call —
# document why you picked them in your report.
from sklearn.preprocessing import MinMaxScaler
 
signals = ['is_exact_dup', 'is_near_dup', 'is_burst_day', 'is_one_hit_wonder', 'rating_deviation']
scaler = MinMaxScaler()
df[signals] = scaler.fit_transform(df[signals])
 
weights = {'is_exact_dup': 0.3, 'is_near_dup': 0.25, 'is_burst_day': 0.2,
           'is_one_hit_wonder': 0.15, 'rating_deviation': 0.1}
df['fake_score'] = sum(df[s] * w for s, w in weights.items())
 
# Threshold into binary pseudo_label — inspect the distribution before picking the cutoff
df['fake_score'].describe()
df['pseudo_label'] = (df['fake_score'] > df['fake_score'].quantile(0.85)).astype(int)  # top 15% flagged as "likely fake"
print(df['pseudo_label'].value_counts())



pseudo_label
0    72833
1    11171
Name: count, dtype: int64


Deliverable

In [ ]:
feature_cols = ['content', 'content_clean', 'score', 'is_exact_dup', 'is_near_dup', 'is_burst_day',
                'is_one_hit_wonder', 'rating_deviation', 'exclamation_count', 'word_count',
                'avg_word_len', 'superlative_count', 'fake_score', 'pseudo_label']
df[feature_cols].to_csv('../data/processed/labeled_reviews.csv', index=False) 
